# RK1828 YOLO26-seg 实例分割 Demo

Ultralytics **YOLO26-seg**：检测框 + 80 类分类 + **像素级实例掩码**，10 个输出头：

| 输出头 | 形状 | 含义 |
|---|---|---|
| `box_*` ×3 | `[1,4,80,80]` `[1,4,40,40]` `[1,4,20,20]` | 框回归 (l,t,r,b) |
| `score_*` ×3 | `[1,80,…]` | 80 类置信度 |
| `mask 系数` ×3 | `[1,32,…]` | 每个候选的 32 维掩码系数 |
| `proto` | `[1,32,160,160]` | 掩码基（prototypes） |

实例掩码 = `sigmoid(32 维系数 · proto)`，再按检测框裁剪、还原到原图。
（和 03 FastSAM 同一套思路，只是多了 80 类分类。）

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 选择模型和图片 |
| 3 | 前处理（letterbox 等比缩放 + 灰边填充） |
| 4 | RKNN 推理 |
| 5 | 后处理（解码 + NMS + 掩码合成） |
| 6 | 可视化 |

> 参考成绩：yolo26m-seg i8 约 250ms/帧，示例图检出 4 个人 + 1 辆巴士。

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


本 demo 下载 YOLO26-seg n/s/m（各 i8/fp16 两版）、COCO 标签和示例图 bus.jpg。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_i8.rknn", "models/yolo26_seg/yolo26n-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_i8.weight", "models/yolo26_seg/yolo26n-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_fp16.rknn", "models/yolo26_seg/yolo26n-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26n-seg_fp16.weight", "models/yolo26_seg/yolo26n-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_i8.rknn", "models/yolo26_seg/yolo26s-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_i8.weight", "models/yolo26_seg/yolo26s-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_fp16.rknn", "models/yolo26_seg/yolo26s-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26s-seg_fp16.weight", "models/yolo26_seg/yolo26s-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_i8.rknn", "models/yolo26_seg/yolo26m-seg_i8.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_i8.weight", "models/yolo26_seg/yolo26m-seg_i8.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_fp16.rknn", "models/yolo26_seg/yolo26m-seg_fp16.rknn"),
    ("yolo26-segment-rknn3-rk1828", "model/yolo26m-seg_fp16.weight", "models/yolo26_seg/yolo26m-seg_fp16.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/coco_80_labels_list.txt", "datasets/coco_80_labels_list.txt"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "datasets/bus.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 选择模型和图片

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(
    options=[f"yolo26{m}-seg_{q}" for m in ("n", "s", "m") for q in ("i8", "fp16")],
    value="yolo26m-seg_i8", description="模型:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="图片:", layout=widgets.Layout(width="60%"))
if any(o.endswith("/bus.jpg") for o in w_image.options):
    w_image.value = [o for o in w_image.options if o.endswith("/bus.jpg")][0]
display(w_model, w_image)

LABELS = open("datasets/coco_80_labels_list.txt").read().splitlines()
print(f"{len(LABELS)} 个 COCO 类别。选好后继续往下运行 ▼")

## 3. 前处理：letterbox

等比缩放 + 灰边填充到 640×640，记下缩放比例和 padding 供坐标还原。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"原图 {img.size} -> 输入 {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# 前处理每个阶段都画出来：原图 -> letterbox -> 输入张量
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. 原图 {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(等比缩放 + 灰边填充)"),
    (axes[2], x[0], f"3. 模型输入 {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN 推理

In [ ]:
import time
from rknn3 import Rknn3Model

model = Rknn3Model(f"models/yolo26_seg/{w_model.value}.rknn")
print(f"{len(model.output_attrs)} 个输出头:")
for a in model.output_attrs:
    print("  ", a)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"推理耗时: {(time.time()-t0)*1000:.1f} ms")

## 5. 后处理：解码 + NMS + 掩码合成

1. 每个网格解码框和 80 类置信度，收集该网格的 32 维掩码系数
2. 按类 NMS 去重
3. `sigmoid(系数 · proto)` 得到 160×160 掩码 → 放大到输入尺寸、按框裁剪 → 还原回原图

In [ ]:
from vision_post import (decode_yolo26_seg, scale_masks, unletterbox_boxes)

boxes, classes, scores, masks = decode_yolo26_seg(outputs, model.output_attrs,
                                                  conf_thresh=0.4)
masks_full = scale_masks(masks, boxes, (img.height, img.width), SCALE, PAD_X, PAD_Y)
boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)

print(f"检测到 {len(boxes)} 个目标:")
for b, c, s in zip(boxes, classes, scores):
    print(f"  {LABELS[c]:12s} {s:.2f}  box=({b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f})")

## 6. 可视化：掩码叠加 + 检测框

In [ ]:
from vision_post import draw_boxes, draw_masks

overlay = draw_masks(img, masks_full)
result = draw_boxes(overlay, boxes, classes, scores, LABELS)
model.close()
result

## 小结

- 掩码合成 `sigmoid(系数 · proto)` 是 YOLOv8-seg / FastSAM / YOLO26-seg 的通用做法，
  proto 只需算一次，所有实例共享。
- 解码在 `vision_post.decode_yolo26_seg`，掩码映射在 `vision_post.scale_masks`。
- 与 03 FastSAM 对比：FastSAM 类别无关（segment everything），YOLO26-seg 输出 80 类。